# Excel 원장 출력 모듈

**입력**: `data/master_gl_clean.csv`  
**출력**: `output/원장_조서_YYYYMMDD.xlsx`

---
**시트 구성**
- `원장` : 선택 계정의 전표 전체 행 (AutoFilter + Freeze)
- `월별집계` : 월별 차변합 / 대변합 / 순잔액 / 누계잔액
- `유형별집계` : 전표유형 × 플래그 교차 집계 (선택 실행)

**사용법**: Cell 1(CONFIG)만 수정 후 전체 실행(Run All)

In [ ]:
# Cell 0 — 라이브러리
import pandas as pd
from pathlib import Path
from datetime import date
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side, NamedStyle
from openpyxl.utils import get_column_letter

print('openpyxl version:', openpyxl.__version__)

In [ ]:
# Cell 1 — ★ CONFIG ★  이 셀만 수정하세요

# ── 계정 선택 ─────────────────────────────────────────────
# 단일: ["500010"]  복수: ["500010", "500020"]  전체: None
TARGET_ACCOUNTS = ["500010"]

# ── lv1/lv2 기준 선택 (TARGET_ACCOUNTS가 None일 때만 작동) ──
# 예: LV1_FILTER = "3"  → lv1_code == '3' 인 계정 전체
#     LV2_FILTER = "500" → lv2_code == '500' 인 계정 전체
LV1_FILTER = None
LV2_FILTER = None

# ── 기간 필터 ──────────────────────────────────────────────
DATE_FROM = "2022-01-01"   # None 이면 전체
DATE_TO   = "2022-12-31"   # None 이면 전체

# ── 집계 단위 ──────────────────────────────────────────────
# "M" = 월별  /  "Q" = 분기별  /  "Y" = 연별
AGG_FREQ = "M"

# ── 경로 ───────────────────────────────────────────────────
DATA_PATH   = Path("../data/master_gl_clean.csv")
MASTER_PATH = Path("../data/account_master.csv")
OUTPUT_DIR  = Path("../output")

# ── 출력 파일명 (None 이면 오늘 날짜 자동) ──────────────────
OUTPUT_NAME = None

In [ ]:
# Cell 2 — 데이터 로드 및 필터링

gl = pd.read_csv(DATA_PATH, dtype={"acc_code": str, "doc_no": str}, low_memory=False)
gl["post_date"] = pd.to_datetime(gl["post_date"])

# 계정 필터
if TARGET_ACCOUNTS:
    mask = gl["acc_code"].isin(TARGET_ACCOUNTS)
elif LV1_FILTER or LV2_FILTER:
    master = pd.read_csv(MASTER_PATH, dtype=str)
    if LV1_FILTER:
        codes = master.loc[master["lv1_code"] == str(LV1_FILTER), "acc_code"]
    else:
        codes = master.loc[master["lv2_code"] == str(LV2_FILTER), "acc_code"]
    mask = gl["acc_code"].isin(codes)
else:
    mask = pd.Series(True, index=gl.index)

# 기간 필터
if DATE_FROM:
    mask &= gl["post_date"] >= DATE_FROM
if DATE_TO:
    mask &= gl["post_date"] <= DATE_TO

filtered = gl[mask].copy().sort_values(["post_date", "doc_no", "line_no"])

print(f"필터 결과: {len(filtered):,} 행 / 계정 {filtered['acc_code'].nunique()}개")
filtered[["doc_no", "post_date", "acc_code", "acc_name", "dc_indicator", "amount"]].head()

In [ ]:
# Cell 3 — Sheet1: 원장 데이터 준비

# 출력할 컬럼 선택 및 한글 헤더 매핑
LEDGER_COLS = {
    "doc_no"       : "전표번호",
    "post_date"    : "전기일",
    "fisc_month"   : "회계연월",
    "acc_code"     : "계정코드",
    "acc_name"     : "계정명",
    "dc_indicator" : "차대구분",
    "_debit"       : "차변금액",
    "_credit"      : "대변금액",
    "amount"       : "금액(절대)",
    "counterparty" : "거래처",
    "department"   : "부서",
    "description"  : "적요",
    "doc_type"     : "전표유형",
    "_flag_weekend"    : "주말플래그",
    "_flag_reversal"   : "역분개플래그",
}

df_ledger = filtered[list(LEDGER_COLS.keys())].copy()
df_ledger["post_date"] = df_ledger["post_date"].dt.strftime("%Y-%m-%d")
df_ledger.columns = list(LEDGER_COLS.values())

print(f"원장 시트: {len(df_ledger):,} 행 × {len(df_ledger.columns)} 열")
df_ledger.head(3)

In [ ]:
# Cell 4 — Sheet2: 월별집계 데이터 준비

# 집계 키 생성
freq_map = {"M": "%Y-%m", "Q": "%Y-Q", "Y": "%Y"}

temp = filtered.copy()
if AGG_FREQ == "Q":
    temp["_period"] = temp["post_date"].dt.to_period("Q").astype(str)
else:
    temp["_period"] = temp["post_date"].dt.strftime(freq_map[AGG_FREQ])

agg = (
    temp.groupby("_period", sort=True)
    .agg(
        차변합계=("_debit",  "sum"),
        대변합계=("_credit", "sum"),
        건수     =("doc_no",  "count"),
    )
    .reset_index()
    .rename(columns={"_period": "기간"})
)

agg["순잔액"]   = agg["차변합계"] - agg["대변합계"]
agg["누계잔액"] = agg["순잔액"].cumsum()

print(agg.to_string(index=False))

In [ ]:
# Cell 5 — Sheet3: 전표유형별 집계 (선택 실행)

agg_type = (
    filtered.groupby(["doc_type"], sort=False)
    .agg(
        건수         =("doc_no",           "count"),
        차변합계     =("_debit",            "sum"),
        대변합계     =("_credit",           "sum"),
        주말건수     =("_flag_weekend",     "sum"),
        역분개건수   =("_flag_reversal",    "sum"),
        중복건수     =("_flag_duplicate",   "sum"),
    )
    .reset_index()
    .rename(columns={"doc_type": "전표유형"})
    .sort_values("차변합계", ascending=False)
)

print(agg_type.to_string(index=False))

In [ ]:
# Cell 6 — Excel 스타일 정의

# 색상 상수
COLOR_HEADER = "1F3864"   # 진남색
COLOR_SUBTOT = "D9E1F2"   # 연파랑 (합계 행)
COLOR_FLAG   = "FCE4D6"   # 연주황 (플래그 열)

thin = Side(border_style="thin", color="AAAAAA")
border_thin = Border(left=thin, right=thin, top=thin, bottom=thin)

def style_header(ws, row=1):
    """헤더 행: 진남색 배경 + 흰 볼드 + 가운데 정렬"""
    for cell in ws[row]:
        cell.font      = Font(bold=True, color="FFFFFF")
        cell.fill      = PatternFill("solid", fgColor=COLOR_HEADER)
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border    = border_thin

def style_total_row(ws, row):
    """합계 행: 연파랑 배경 + 볼드"""
    for cell in ws[row]:
        cell.font   = Font(bold=True)
        cell.fill   = PatternFill("solid", fgColor=COLOR_SUBTOT)
        cell.border = border_thin

def set_col_widths(ws, widths: dict):
    """widths: {col_letter: width} 또는 자동 추정"""
    for col_idx, col_cells in enumerate(ws.columns, start=1):
        max_len = max((len(str(c.value)) if c.value else 0) for c in col_cells)
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 4, 40)

def fmt_amount_col(ws, col_letter, start_row=2):
    """금액 열 천단위 서식"""
    for row in ws.iter_rows(min_row=start_row, min_col=ws[col_letter + str(start_row)].column,
                             max_col=ws[col_letter + str(start_row)].column):
        for cell in row:
            cell.number_format = "#,##0"

print("스타일 함수 정의 완료")

In [ ]:
# Cell 7 — Excel 파일 생성 (Sheet1: 원장)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fname = OUTPUT_NAME or f"원장_조서_{date.today().strftime('%Y%m%d')}.xlsx"
out_path = OUTPUT_DIR / fname

wb = openpyxl.Workbook()

# ── Sheet1: 원장 ──────────────────────────────────────────
ws1 = wb.active
ws1.title = "원장"

# 헤더 쓰기
ws1.append(list(df_ledger.columns))

# 데이터 쓰기
for row in df_ledger.itertuples(index=False):
    ws1.append(list(row))

# 헤더 스타일
style_header(ws1, row=1)

# 플래그 열 배경색
flag_cols = ["주말플래그", "역분개플래그"]
flag_col_idxs = [df_ledger.columns.get_loc(c) + 1 for c in flag_cols if c in df_ledger.columns]
for col_idx in flag_col_idxs:
    for row_idx in range(2, ws1.max_row + 1):
        cell = ws1.cell(row=row_idx, column=col_idx)
        if cell.value:
            cell.fill = PatternFill("solid", fgColor=COLOR_FLAG)

# 금액 열 서식
amt_cols = ["차변금액", "대변금액", "금액(절대)"]
for col_name in amt_cols:
    if col_name in df_ledger.columns:
        col_idx = df_ledger.columns.get_loc(col_name) + 1
        col_letter = get_column_letter(col_idx)
        for row_idx in range(2, ws1.max_row + 1):
            ws1.cell(row=row_idx, column=col_idx).number_format = "#,##0"

# AutoFilter + Freeze
ws1.auto_filter.ref = ws1.dimensions
ws1.freeze_panes = "A2"

# 열 너비 자동 조정
set_col_widths(ws1, {})

print(f"Sheet1 '원장' 완료 — {ws1.max_row - 1:,} 행")

In [ ]:
# Cell 8 — Excel 파일 생성 (Sheet2: 월별집계)

ws2 = wb.create_sheet("월별집계")

# 헤더
headers2 = list(agg.columns)
ws2.append(headers2)

# 데이터
for row in agg.itertuples(index=False):
    ws2.append(list(row))

# 합계 행 (Excel SUM 수식 삽입)
data_start = 2
data_end   = ws2.max_row
sum_row    = data_end + 1

ws2.cell(sum_row, 1).value = "합계"

sum_cols = {"차변합계": 2, "대변합계": 3, "건수": 4, "순잔액": 5}
for col_name, col_idx in sum_cols.items():
    col_letter = get_column_letter(col_idx)
    ws2.cell(sum_row, col_idx).value = f"=SUM({col_letter}{data_start}:{col_letter}{data_end})"

# 누계잔액 합계는 의미 없으므로 빈칸

# 스타일
style_header(ws2, row=1)
style_total_row(ws2, row=sum_row)

# 금액 열 서식
for col_idx in [2, 3, 5, 6]:  # 차변합계, 대변합계, 순잔액, 누계잔액
    for row_idx in range(2, sum_row + 1):
        ws2.cell(row=row_idx, column=col_idx).number_format = "#,##0"

ws2.freeze_panes = "A2"
set_col_widths(ws2, {})

print(f"Sheet2 '월별집계' 완료 — {data_end - 1} 기간 + 합계 행")

In [ ]:
# Cell 9 — Excel 파일 생성 (Sheet3: 유형별집계)

ws3 = wb.create_sheet("유형별집계")

headers3 = list(agg_type.columns)
ws3.append(headers3)

for row in agg_type.itertuples(index=False):
    ws3.append(list(row))

# 합계 행
data_end3 = ws3.max_row
sum_row3  = data_end3 + 1
ws3.cell(sum_row3, 1).value = "합계"

num_cols3 = {2: "건수", 3: "차변합계", 4: "대변합계", 5: "주말건수", 6: "역분개건수", 7: "중복건수"}
for col_idx in num_cols3:
    col_letter = get_column_letter(col_idx)
    ws3.cell(sum_row3, col_idx).value = f"=SUM({col_letter}2:{col_letter}{data_end3})"

style_header(ws3, row=1)
style_total_row(ws3, row=sum_row3)

# 금액/건수 서식
for col_idx in range(2, 8):
    for row_idx in range(2, sum_row3 + 1):
        ws3.cell(row=row_idx, column=col_idx).number_format = "#,##0"

# 플래그 열 경고색
for col_idx in [5, 6, 7]:  # 주말/역분개/중복 건수
    for row_idx in range(2, data_end3 + 1):
        cell = ws3.cell(row=row_idx, column=col_idx)
        if cell.value and cell.value > 0:
            cell.fill = PatternFill("solid", fgColor=COLOR_FLAG)

ws3.freeze_panes = "A2"
set_col_widths(ws3, {})

print(f"Sheet3 '유형별집계' 완료 — {data_end3} 유형")

In [ ]:
# Cell 10 — 저장

wb.save(out_path)
print(f"✓ 저장 완료: {out_path.resolve()}")